# Stage 03 / Lesson 08: In-Memory Vector Store (`InMemoryVectorStore`)
**Course Lecture:** Lecture 34: InMemory Vector Store  
**Section:** Section 7: Vector Stores and Vector Databases

---

## 🧭 Architectural Overview & Concept Summary
The **`InMemoryVectorStore`** is the standard, zero-dependency, reference in-memory vector store provided natively by **`langchain-core`**.

### Key Architectural Characteristics:
1. **Zero External Dependencies:** Built entirely with standard Python data structures (`dict`) and NumPy. Requires no local C++ toolchains (unlike FAISS binaries), no SQLite databases (unlike Chroma), and zero cloud credentials.
2. **Process-Memory Bound (Ephemeral):** Vectors and document payloads reside strictly within application heap memory (RAM). When the host process terminates, all indexed content is discarded.
3. **Exact Nearest Neighbor (Linear Scan):** Uses NumPy to compute vector dot products and $L_2$ norms for exact cosine similarity across all stored items:
   $$\text{Cosine Similarity}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|_2 \|\mathbf{v}\|_2}$$
4. **Primary Engineering Use-Cases:**
   - **Automated Testing & CI/CD Pipelines:** Rapid unit testing for RAG pipelines without provisioning disk artifacts or network mocks.
   - **Ephemeral Session Knowledge:** Chatbot sessions where documents uploaded by a user are only relevant for the duration of the current WebSocket connection.
   - **Low-Latency Micro-Corpora:** Small datasets (< 10,000 vectors) where the latency overhead of network calls to cloud vector databases exceeds local linear scan times.

## 1. Setup & Environment Initialization
Load API credentials from `.env` and verify installation of `langchain-core` components.

In [1]:
import os
import sys
import numpy as np
from pathlib import Path
from dotenv import load_dotenv

# load environment variables
load_dotenv()

# import in-memory vector store and core document interfaces
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

# initialize local embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# instantiate in-memory vector store
vector_store = InMemoryVectorStore(embeddings)

print("✅ InMemoryVectorStore initialized successfully!")
print(f"📦 Embedding dimension: {len(embeddings.embed_query('test'))}")

✅ InMemoryVectorStore initialized successfully!
📦 Embedding dimension: 384


## 2. Ingesting Multi-Domain Corpus with Rich Metadata
Construct an in-memory knowledge base containing documents across multiple categories (daily activities, weather reports, artificial intelligence engineering) accompanied by structured metadata.

In [2]:
# construct multi-domain document collection
sample_documents = [
    Document(
        page_content="I had chocolate chips and scrambled eggs for breakfast this morning.",
        metadata={"source": "tweet", "topic": "food", "author": "user_42"}
    ),
    Document(
        page_content="The weather forecast for tomorrow is cloudy with scattered rain showers.",
        metadata={"source": "news", "topic": "weather", "urgency": "low"}
    ),
    Document(
        page_content="Building RAG pipelines with LangChain and LCEL allows clean, modular, and streamable prompt chaining.",
        metadata={"source": "tech_blog", "topic": "ai", "tier": "free"}
    ),
    Document(
        page_content="Severe weather warnings were issued for coastal areas due to an approaching tropical storm.",
        metadata={"source": "news", "topic": "weather", "urgency": "high"}
    ),
    Document(
        page_content="A balanced breakfast rich in protein and fiber provides sustained energy throughout the day.",
        metadata={"source": "health_article", "topic": "food", "tier": "free"}
    ),
    Document(
        page_content="Vector databases index dense embeddings to enable sub-millisecond semantic search at massive scale.",
        metadata={"source": "tech_blog", "topic": "ai", "tier": "enterprise"}
    ),
]

# add documents to the in-memory vector store
doc_ids = vector_store.add_documents(documents=sample_documents)

print(f"✅ Ingested {len(doc_ids)} documents into InMemoryVectorStore.")
for idx, (doc_id, doc) in enumerate(zip(doc_ids, sample_documents), start=1):
    print(f"  [{idx}] ID: {doc_id} | Topic: {doc.metadata['topic']:<8} | Snippet: '{doc.page_content[:50]}...'")

✅ Ingested 6 documents into InMemoryVectorStore.
  [1] ID: c5b9de97-63ff-45ed-8d74-237279fcf89a | Topic: food     | Snippet: 'I had chocolate chips and scrambled eggs for break...'
  [2] ID: 53588e95-d4b1-4644-8b36-875e24babdb3 | Topic: weather  | Snippet: 'The weather forecast for tomorrow is cloudy with s...'
  [3] ID: d737c306-bdd8-4d19-8773-2e3dcdb4cb24 | Topic: ai       | Snippet: 'Building RAG pipelines with LangChain and LCEL all...'
  [4] ID: 657c8cc3-8e00-427f-9625-22bc49386846 | Topic: weather  | Snippet: 'Severe weather warnings were issued for coastal ar...'
  [5] ID: abe383a6-6405-4f46-b09d-1cdf86a0d00d | Topic: food     | Snippet: 'A balanced breakfast rich in protein and fiber pro...'
  [6] ID: 4411878c-9731-4497-89c3-65923f5e7e9b | Topic: ai       | Snippet: 'Vector databases index dense embeddings to enable ...'


## 3. Semantic Similarity Search Analysis
Perform semantic similarity search against the in-memory collection and evaluate retrieval results.

In [3]:
# execute standard semantic similarity query
query = "How is the weather forecast looking?"
results = vector_store.similarity_search(query, k=2)

print(f"🔍 Query: '{query}'\n")
print(f"Top {len(results)} Retrieved Documents:")
for idx, doc in enumerate(results, start=1):
    print(f"  [{idx}] Topic: {doc.metadata['topic']} | Source: {doc.metadata['source']}")
    print(f"      Content: '{doc.page_content}'\n")

🔍 Query: 'How is the weather forecast looking?'

Top 2 Retrieved Documents:
  [1] Topic: weather | Source: news
      Content: 'The weather forecast for tomorrow is cloudy with scattered rain showers.'

  [2] Topic: weather | Source: news
      Content: 'Severe weather warnings were issued for coastal areas due to an approaching tropical storm.'


## 4. Similarity Scoring Evaluation (Cosine Metric)
Evaluate retrieval confidence using `similarity_search_with_score`.  
Unlike distance-based stores (such as FAISS L2 distance where 0 is exact match), `InMemoryVectorStore` yields normalized cosine similarity scores in the range $[-1.0, 1.0]$ (where higher indicates greater semantic proximity).

In [4]:
# retrieve document chunks along with cosine similarity scores
scored_results = vector_store.similarity_search_with_score(query, k=4)

print(f"🔍 Query: '{query}'\n")
print("Ranked Cosine Similarity Distribution (Higher score = closer semantic match):")
for idx, (doc, score) in enumerate(scored_results, start=1):
    print(f"  Rank #{idx} | Score: {score:.4f}")
    print(f"    Topic:   {doc.metadata['topic']:<8} | Source: {doc.metadata['source']}")
    print(f"    Snippet: {doc.page_content}\n")

🔍 Query: 'How is the weather forecast looking?'

Ranked Cosine Similarity Distribution (Higher score = closer semantic match):
  Rank #1 | Score: 0.5841
    Topic:   weather  | Source: news
    Snippet: The weather forecast for tomorrow is cloudy with scattered rain showers.

  Rank #2 | Score: 0.4304
    Topic:   weather  | Source: news
    Snippet: Severe weather warnings were issued for coastal areas due to an approaching tropical storm.

  Rank #3 | Score: 0.1076
    Topic:   food     | Source: tweet
    Snippet: I had chocolate chips and scrambled eggs for breakfast this morning.

  Rank #4 | Score: 0.0685
    Topic:   food     | Source: health_article
    Snippet: A balanced breakfast rich in protein and fiber provides sustained energy throughout the day.


## 5. Functional Metadata Filtering
`InMemoryVectorStore` supports callable / lambda predicates for runtime metadata filtering without SQL-style syntax constraints.

In [5]:
# define dynamic callable filter: only select high-urgency weather documents
high_urgency_filter = lambda doc: doc.metadata.get("urgency") == "high"

filtered_weather = vector_store.similarity_search(
    query="Are there any severe weather threats?",
    k=2,
    filter=high_urgency_filter
)

print(f"Filtered Results (urgency == 'high'): {len(filtered_weather)}")
for idx, doc in enumerate(filtered_weather, start=1):
    print(f"  [{idx}] Urgency: {doc.metadata.get('urgency')} | Content: {doc.page_content}")

# define secondary filter: only select free tech articles
tech_free_filter = lambda doc: doc.metadata.get("topic") == "ai" and doc.metadata.get("tier") == "free"
filtered_tech = vector_store.similarity_search(
    query="How does LangChain help with retrieval?",
    k=2,
    filter=tech_free_filter
)

print(f"\nFiltered Results (topic == 'ai' and tier == 'free'): {len(filtered_tech)}")
for idx, doc in enumerate(filtered_tech, start=1):
    print(f"  [{idx}] Tier: {doc.metadata.get('tier')} | Content: {doc.page_content}")

Filtered Results (urgency == 'high'): 1
  [1] Urgency: high | Content: Severe weather warnings were issued for coastal areas due to an approaching tropical storm.

Filtered Results (topic == 'ai' and tier == 'free'): 1
  [1] Tier: free | Content: Building RAG pipelines with LangChain and LCEL allows clean, modular, and streamable prompt chaining.


## 6. Full LCEL RAG Pipeline Integration
Construct an end-to-end question-answering pipeline using LangChain Expression Language (LCEL), the in-memory retriever, and Groq's high-speed inference engine.

In [6]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. create retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# 2. formatting utility
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 3. grounded prompt template
template = """You are a precise technical AI assistant.
Answer the user's question using ONLY the provided context. If the answer cannot be determined from the context, state "I don't know."

Context:
{context}

Question:
{question}

Answer:"""
prompt = ChatPromptTemplate.from_template(template)

# 4. groq chat model
llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=512
)

# 5. assemble lcel chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 6. execute verification query
rag_query = "What is the weather forecast for tomorrow and what precautions were issued?"
response = rag_chain.invoke(rag_query)

print(f"❓ Question: {rag_query}\n")
print(f"💬 Generated Answer:\n{response}")

❓ Question: What is the weather forecast for tomorrow and what precautions were issued?

💬 Generated Answer:
Tomorrow will be cloudy with scattered rain showers, and severe weather warnings were issued for coastal areas due to an approaching tropical storm.


## 7. Token-by-Token Streaming Demonstration
Verify native real-time token streaming supported by the LCEL pipeline.

In [7]:
streaming_query = "What did someone eat for breakfast and what makes breakfast healthy?"
print(f"❓ Question: {streaming_query}\n")
print("🌊 Streaming Response: ")

for chunk in rag_chain.stream(streaming_query):
    print(chunk, end="", flush=True)
print("\n\n[Streaming Completed!]")

❓ Question: What did someone eat for breakfast and what makes breakfast healthy?

🌊 Streaming Response: 
They ate chocolate chips and scrambled eggs. A balanced breakfast that is rich in protein and fiber provides sustained energy throughout the day.

[Streaming Completed!]


## 8. Architectural Comparison Matrix
| Architectural Feature | `InMemoryVectorStore` | `FAISS` (Local Index) | `ChromaDB` (Persistent) | `Pinecone` (Cloud DB) |
| :--- | :--- | :--- | :--- | :--- |
| **Storage Medium** | Heap RAM (`dict` + NumPy) | Process RAM (`IndexFlatL2` C++) | SQLite + Parquet on Disk | Managed Distributed Cloud |
| **Persistence** | None (Lost on exit) | Manual (`save_local` / `load_local`) | Automatic on Disk | Cloud-native High Availability |
| **Dependencies** | Pure `langchain-core` | `faiss-cpu` (C++ binaries) | `chromadb` (SQLite, C++ wheels) | `pinecone` client (API-only) |
| **Search Algorithm** | Brute-force Exact Scan | Exact Flat or Approximate (IVF, HNSW) | HNSW Approximate Search | Proprietary Cloud HNSW |
| **Ideal Workload** | Unit tests, ephemeral chat sessions (< 10k vectors) | Local high-speed search (< 1M vectors) | Embedded desktop/offline apps (< 500k vectors) | Enterprise scale (> 10M vectors, multi-tenant) |